# TFM — Gastronomía asiática en Europa
## Semana 3 (extra): Red densa (MLP) con Keras — módulo de Deep Learning

Quinto modelo de la comparativa, para conectar el TFM con lo practicado en la asignatura de Deep Learning
(regresión tabular con red densa, Dropout y regularización L2 — mismo patrón que la tarea de esa asignatura
sobre el dataset California Housing).

**Nota técnica:** este notebook se ejecuta con un kernel de Python distinto al resto del TFM
(entorno virtual `tfm-tf`, ver `../.venv_tf`), porque TensorFlow provocaba un `segmentation fault`
al importarse en el entorno base de Anaconda usado en las demás semanas (conflicto de versión de
`protobuf` con otros paquetes ya instalados ahí, como streamlit). Para no arriesgar la estabilidad del
entorno principal se creó este entorno aislado solo con las librerías necesarias
(`pandas`, `numpy`, `scikit-learn`, `tensorflow`).

Parte de los mismos datos preprocesados y la misma partición train/test (`random_state=42`) que
`03_preprocesamiento_modelos.ipynb`, para que los resultados sean directamente comparables.


In [1]:
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

print('TensorFlow', tf.__version__)

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)


Matplotlib is building the font cache; this may take a moment.


TensorFlow 2.21.0


## 1. Cargar el dataset preprocesado (Semana 3) y reconstruir el mismo split

In [2]:
df = pd.read_csv('../data/asian_restaurants_preprocessed.csv', low_memory=False)
print('Shape:', df.shape)

target = 'avg_rating'

numeric_features = ['log_reviews_count', 'food', 'service', 'value', 'atmosphere',
                    'open_days_per_week', 'open_hours_per_week', 'working_shifts_per_week',
                    'latitude', 'longitude', 'price_level_ord', 'n_cuisines']

binary_features = ['claimed_flag', 'vegetarian_flag', 'vegan_flag', 'gluten_free_flag'] + \
    [c for c in df.columns if c.startswith('meal_')] + \
    [c for c in df.columns if c.startswith('cuisine_')]

categorical_features = ['country_grouped', 'region']

feature_cols = numeric_features + binary_features + categorical_features

X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Train:', X_train.shape, ' Test:', X_test.shape)


Shape: (81315, 40)
Train: (65052, 35)  Test: (16263, 35)


## 2. Preprocesamiento (idéntico al notebook principal, pero salida densa)

Keras necesita arrays numéricos densos, no matrices dispersas: se usa `OneHotEncoder(sparse_output=False)`.

In [3]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('bin', binary_transformer, binary_features),
    ('cat', categorical_transformer, categorical_features),
])

X_train_proc = preprocessor.fit_transform(X_train).astype('float32')
X_test_proc = preprocessor.transform(X_test).astype('float32')
y_train_arr = y_train.values.astype('float32')
y_test_arr = y_test.values.astype('float32')

print('X_train_proc shape:', X_train_proc.shape)


X_train_proc shape: (65052, 294)


## 3. Arquitectura — mismo patrón que la tarea de Deep Learning

Red densa con Dropout + regularización L2 (las dos técnicas de regularización combinadas en la tarea
de la asignatura), y `EarlyStopping` para evitar sobreajuste.

In [4]:
n_features = X_train_proc.shape[1]

model = keras.Sequential([
    keras.Input(shape=(n_features,)),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(1, activation='linear'),
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        37,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,129 (188.00 KB)

 Trainable params: 48,129 (188.00 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True,
)

history = model.fit(
    X_train_proc, y_train_arr,
    validation_split=0.15,
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=0,
)

print(f'Entrenamiento detenido en la época {len(history.history["loss"])} (EarlyStopping)')


Entrenamiento detenido en la época 23 (EarlyStopping)


## 4. Curvas de entrenamiento

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss (MSE)')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(history.history['mae'], label='train')
axes[1].plot(history.history['val_mae'], label='val')
axes[1].set_title('MAE')
axes[1].set_xlabel('Época')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/fig_keras_training_curves.png', dpi=100)
plt.show()


/var/folders/pq/lmth_7lx2klcsmjb8j4zc1_m0000gn/T/ipykernel_70610/3530210057.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Evaluación sobre el conjunto de test

In [7]:
y_pred = model.predict(X_test_proc, verbose=0).flatten()

rmse = float(np.sqrt(mean_squared_error(y_test_arr, y_pred)))
mae = float(mean_absolute_error(y_test_arr, y_pred))
r2 = float(r2_score(y_test_arr, y_pred))

print(f'RMSE={rmse:.4f}  MAE={mae:.4f}  R2={r2:.4f}')


RMSE=0.5404  MAE=0.3618  R2=0.3886


## 6. Guardar métricas para la tabla comparativa del notebook principal

In [8]:
metrics = {'RMSE': rmse, 'MAE': mae, 'R2': r2}

with open('../data/dense_nn_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Guardado en ../data/dense_nn_metrics.json:', metrics)


Guardado en ../data/dense_nn_metrics.json: {'RMSE': 0.5404396433959258, 'MAE': 0.3618367910385132, 'R2': 0.388589084148407}
